# SAM 3D Body with OpenVINO on Intel GPU

[SAM 3D Body](https://github.com/facebookresearch/sam-3d-body) is Meta's 3D human
pose and shape estimator: given an image and a person box it predicts 2D/3D
keypoints and a full parametric body mesh. This notebook takes the reference
PyTorch checkpoint, capable of running on CPU and Intel XPU as chosen, converts it to
OpenVINO IR (FP16 and INT8), runs the converted model on the Intel GPU,
and verifies that the outputs match — visually and by PCK@0.05 metric.

```
                 ┌────────────────────────┐
  Human image ──▶│ PyTorch  (CPU or XPU)  │──▶ keypoints + mesh + PCK   ← reference
                 └───────────┬────────────┘
                             │  torch.jit.trace ─▶ ov.convert_model
                             ▼
                 ┌────────────────────────┐
                 │  OpenVINO IR  FP16     │──▶ keypoints + mesh + PCK
                 │  OpenVINO IR  INT8     │──▶ keypoints + mesh + PCK
                 └────────────────────────┘
                          on Intel GPU
```

**Model pipeline** (per person):
`backbone (DINOv3 ViT-H/16+)` → `iterative decoder ×6 with MHR feedback` → `MHR mesh head` → `2D/3D keypoints`

The three helper modules and the bundled sample image live in this folder. The
`sam_3d_body` model package is fetched at runtime from the original SAM 3D Body's
official [GitHub repo](https://github.com/facebookresearch/sam-3d-body) and placed
in this same folder.

| File | Responsibility | Requires |
|---|---|---|
| [`sam3d_data.py`](sam3d_data.py) | sample loading, PCK scoring, skeleton + mesh rendering | NumPy, OpenCV, Matplotlib, pyrender, trimesh |
| [`sam3d_ov.py`](sam3d_ov.py) | OpenVINO IR runtime (iterative decoder loop) | OpenVINO, NumPy, OpenCV |
| [`sam3d_torch.py`](sam3d_torch.py) | PyTorch reference inference + PyTorch → OpenVINO export | PyTorch |
| `sample_data/` | the demo image and its ground-truth annotation | — |

> Every section is re-runnable, and work that is already done (package fetched,
> image downloaded, checkpoint fetched, IR exported) is **skipped automatically**.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Configuration and device discovery](#Configuration-and-device-discovery)
- [Download the reference PyTorch model](#Download-the-reference-PyTorch-model)
- [Sample image and ground truth](#Sample-image-and-ground-truth)
- [PyTorch reference inference](#PyTorch-reference-inference)
- [Convert the model to OpenVINO IR](#Convert-the-model-to-OpenVINO-IR)
- [Run OpenVINO inference on Intel GPU](#Run-OpenVINO-inference-on-Intel-GPU)
- [Compare PyTorch and OpenVINO](#Compare-PyTorch-and-OpenVINO)
- [Summary](#Summary)

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

This notebook runs in a dedicated **conda environment**. Create it once from a
terminal, register it as a Jupyter kernel, then select that kernel here:

```bash
conda create -n sam3dbody-nb python=3.11 -y
conda activate sam3dbody-nb

# Install necessary packages (Pytorch: CPU-only). Add the --extra-index-url flag for Pytorch: XPU.
pip install -r requirements.txt
# pip install -r requirements.txt --extra-index-url https://download.pytorch.org/whl/xpu

python -m ipykernel install --user --name sam3dbody-nb --display-name "Python (sam3dbody-nb)"
```

Then pick Python **(sam3dbody-nb)** from the kernel picker in the top-right corner.

The cell below confirms which environment the kernel is actually running in and
installs anything still missing *into that same environment*. Set
`FORCE_REINSTALL = True` to reinstall the full requirements file.

In [ ]:
CONDA_ENV_NAME = "sam3dbody-nb"   # environment this notebook expects
FORCE_REINSTALL = False        # True -> always install the full requirements file

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

active_env = os.environ.get("CONDA_DEFAULT_ENV") or Path(sys.prefix).name
print(f"Python      : {sys.version.split()[0]}")
print(f"Interpreter : {sys.executable}")
print(f"Conda env   : {active_env}")

if active_env != CONDA_ENV_NAME:
    print(
        f"\nWARNING: this kernel is running in '{active_env}', not '{CONDA_ENV_NAME}'.\n"
        f"         Follow the steps above and select the 'Python ({CONDA_ENV_NAME})' kernel."
    )

REQ_FILE = Path("requirements.txt")
if not REQ_FILE.exists():  # notebook launched from the repository root
    REQ_FILE = Path("notebooks/requirements.txt")

# pip name -> importable module name (they differ for a few packages)
MODULES = {
    "numpy": "numpy", "opencv-python": "cv2", "scipy": "scipy", "Pillow": "PIL",
    "torch": "torch", "openvino": "openvino", "nncf": "nncf",
    "timm": "timm", "einops": "einops", "omegaconf": "omegaconf",
    "hydra-core": "hydra", "roma": "roma", "yacs": "yacs",
    "matplotlib": "matplotlib", "trimesh": "trimesh", "pyrender": "pyrender",
    "tqdm": "tqdm", "huggingface_hub": "huggingface_hub", "ipykernel": "ipykernel",
}

missing = [pkg for pkg, mod in MODULES.items() if importlib.util.find_spec(mod) is None]

if FORCE_REINSTALL or missing:
    args = ["-r", str(REQ_FILE)] if FORCE_REINSTALL else missing
    print("\nInstalling into the active environment:", " ".join(args))
    # The XPU wheel index is harmless for non-torch packages and required for torch+xpu.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *args,
         "--extra-index-url", "https://download.pytorch.org/whl/xpu"],
        check=True,
    )
else:
    print("\nAll required packages are already installed - nothing to do.")

## Configuration and device discovery
[back to top ⬆️](#Table-of-contents:)

Imports the helper modules and reports which accelerators are visible to
**OpenVINO** and to **PyTorch**.

In [ ]:
import time

# pyrender must pick a headless GL backend *before* it is first imported.
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

# --- Make the helper modules importable, wherever the notebook was launched ---
NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "sam3d_data.py").exists() else Path.cwd() / "notebooks"
sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
import openvino as ov
import torch

import sam3d_data as data
from sam3d_data import COCO17_TO_MHR70, compute_person_pck, free_memory, to_numpy
from sam3d_torch import find_repo_root, xpu_available

# The `sam_3d_body` package is Meta's code under the SAM License. Only that single
# package is fetched at runtime (a sparse clone of ~1 MB, not the whole repo) and
# placed in the root folder, beside these helpers — it is not redistributed with them.
MODEL_SRC = find_repo_root()
# Checkpoints and exported IRs live next to the notebook (see README).
DATA_ROOT = NOTEBOOK_DIR

print(f"Helpers   : {NOTEBOOK_DIR}")
print(f"Model src : {MODEL_SRC}")
print(f"Data root : {DATA_ROOT}")
print(f"NumPy     : {np.__version__}")
print(f"OpenVINO  : {ov.__version__}")
print(f"PyTorch   : {torch.__version__}")

# --- OpenVINO devices ---------------------------------------------------------
core = ov.Core()
print(f"\nOpenVINO devices: {core.available_devices}")
for dev in core.available_devices:
    try:
        print(f"  {dev:5s} -> {core.get_property(dev, 'FULL_DEVICE_NAME')}")
    except Exception:
        print(f"  {dev:5s} -> (name unavailable)")

# --- PyTorch devices ----------------------------------------------------------
print("\nPyTorch devices:")
print("  cpu   -> always available")
print(f"  xpu   -> {torch.xpu.get_device_name(0) if xpu_available() else 'not available'}")

In [ ]:
# =============================== CONFIGURATION ===============================
# Reference PyTorch checkpoint (Hugging Face)
HF_REPO_ID = "facebook/sam-3d-body-dinov3"
CKPT_DIR   = DATA_ROOT / "checkpoints" / "sam-3d-body-dinov3"
CHECKPOINT = CKPT_DIR / "model.ckpt"            # 2.0 GB
MHR_PATH   = CKPT_DIR / "assets" / "mhr_model.pt"   # 664 MB

# Sample: bundled in ./sample_data (the image is downloaded on first use).
SAMPLE_NAME = "000000368212"

# PyTorch reference backend: "cpu" (default, works everywhere) or "xpu" for an
# Intel GPU, but needs a torch+xpu build.
TORCH_DEVICE = "cpu"

# OpenVINO
OV_MODEL_ROOT = DATA_ROOT / "ov_models"            # <root>/{fp16,int8}/...
OV_DEVICE     = "GPU"                  # target accelerator
PRECISIONS    = ["fp16", "int8"]

# Evaluation
PCK_THRESHOLD = 0.05      # 5% of the GT bbox diagonal (paper protocol)
PAPER_PCK     = 86.5      # DINOv3-H+ COCO val2017 PCK@0.05 from the paper

# Set True to re-export the OpenVINO IR even if it already exists (slow)
FORCE_EXPORT = False
# =============================================================================

if OV_DEVICE not in core.available_devices:
    print(f"WARNING: '{OV_DEVICE}' not found in {core.available_devices}; falling back to CPU.")
    OV_DEVICE = "CPU"

if TORCH_DEVICE == "xpu" and not xpu_available():
    print("WARNING: XPU requested but unavailable; the PyTorch stage will use the CPU.")
    TORCH_DEVICE = "cpu"

print(f"PyTorch reference device : {TORCH_DEVICE}")
print(f"OpenVINO target device   : {OV_DEVICE}")
print(f"Precisions to compare    : {PRECISIONS}")

RESULTS = {}  # backend name -> {person, keypoints, pck, latency_ms}

## Download the reference PyTorch model
[back to top ⬆️](#Table-of-contents:)

Pulls `facebook/sam-3d-body-dinov3` from Hugging Face (~2.7 GB total) — skipped
if the checkpoint is already on disk.

> The SAM 3D Body checkpoints are **gated**. Request access on the model page and
> authenticate once with `hf auth login` (or set `HF_TOKEN`) before running this cell.

In [ ]:
def download_checkpoint():
    """Fetch the reference checkpoint from Hugging Face if it is not present."""
    if CHECKPOINT.exists() and MHR_PATH.exists():
        size_gb = (CHECKPOINT.stat().st_size + MHR_PATH.stat().st_size) / 1024**3
        print(f"[skip] Checkpoint already present at {CKPT_DIR}  ({size_gb:.1f} GB)")
        return

    from huggingface_hub import snapshot_download

    print(f"Downloading {HF_REPO_ID} -> {CKPT_DIR} ...")
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id=HF_REPO_ID, local_dir=str(CKPT_DIR))
    print("Download complete.")


download_checkpoint()

for label, path in [("model.ckpt", CHECKPOINT), ("mhr_model.pt", MHR_PATH)]:
    status = f"{path.stat().st_size / 1024**2:8.1f} MB" if path.exists() else "MISSING"
    print(f"  {label:14s} {status}   {path}")

## Sample image and ground truth
[back to top ⬆️](#Table-of-contents:)

The demo image is from COCO val17, ships in [`sample_data/`](sample_data) together with its
ground-truth annotation; if the `.jpg` is missing it is downloaded from the COCO
servers on first use. The image has exactly one person with all 17 keypoints
visible and a large bounding box, which makes for a clean comparison.

The ground-truth box is fed to the model, so this measures the pose accuracy without a detector in the loop.

In [ ]:
SAMPLE    = data.load_sample(SAMPLE_NAME)
IMG_BGR   = SAMPLE["img_bgr"]
IMG_RGB   = SAMPLE["img_rgb"]
GT_ANN    = SAMPLE["annotation"]
BBOX_XYXY = SAMPLE["bbox_xyxy"]
gt_kpts   = SAMPLE["gt_keypoints"]

# With no FOV estimator in the loop both backends default to the image diagonal.
# Pinning it here guarantees PyTorch and OpenVINO use the *same* camera.
FOCAL = SAMPLE["focal_length"]

gx, gy, gw, gh = SAMPLE["bbox_xywh"]
print(f"Image      : {SAMPLE['path']}  ({SAMPLE['width']}x{SAMPLE['height']})")
print(f"Source     : {SAMPLE['source']}")
print(f"GT bbox    : x1={gx:.0f} y1={gy:.0f} x2={gx+gw:.0f} y2={gy+gh:.0f}")
print(f"Visible GT : {int((gt_kpts[:, 2] > 0).sum())}/17 keypoints")
print(f"Focal      : {FOCAL:.1f} px (image diagonal)")

plt.figure(figsize=(6, 6))
plt.imshow(IMG_RGB)
plt.gca().add_patch(plt.Rectangle((gx, gy), gw, gh, fill=False, color="lime", lw=2))
vis = gt_kpts[:, 2] > 0
plt.scatter(gt_kpts[vis, 0], gt_kpts[vis, 1], c="red", s=26, edgecolors="white", linewidths=0.8)
plt.title(f"Sample: {SAMPLE['file_name']}  —  GT bbox (green) + GT keypoints (red)")
plt.axis("off")
plt.tight_layout()
plt.show()

## PyTorch reference inference
[back to top ⬆️](#Table-of-contents:)

`sam3d_torch.Sam3DBodyTorch` runs the checkpoint on the device chosen by
`TORCH_DEVICE`. The model code hard-codes `.cuda()` calls, so the helper
transparently redirects them to `cpu` or `xpu` — the model source itself is
never modified.

In [ ]:
from sam3d_torch import Sam3DBodyTorch

t0 = time.perf_counter()
torch_model = Sam3DBodyTorch(
    checkpoint_path=str(CHECKPOINT), mhr_path=str(MHR_PATH), device=TORCH_DEVICE
)
print(f"\nModel loaded in {time.perf_counter() - t0:.1f}s")

# `faces` is the shared mesh topology used by every renderer below.
MESH_FACES = to_numpy(torch_model.faces)
print(f"Mesh faces: {MESH_FACES.shape}")

In [ ]:
# --- Run inference (the first call includes lazy kernel compilation) ----------
_ = torch_model.infer(str(SAMPLE["path"]), bboxes=BBOX_XYXY[None], use_mask=False)  # warm-up
outputs = torch_model.infer(str(SAMPLE["path"]), bboxes=BBOX_XYXY[None], use_mask=False)

pred = outputs[0]
torch_person = {
    "keypoints_2d": to_numpy(pred["pred_keypoints_2d"]),   # [70, 2]
    "bbox":         np.asarray(pred["bbox"], dtype=np.float32),
    "vertices":     to_numpy(pred["pred_vertices"]),       # [V, 3]
    "cam_t":        to_numpy(pred["pred_cam_t"]),          # [3]
    "focal_length": float(pred["focal_length"]),
}

torch_pck, torch_correct, torch_valid = compute_person_pck(
    torch_person["keypoints_2d"], GT_ANN, PCK_THRESHOLD
)

REFERENCE = f"PyTorch {TORCH_DEVICE.upper()}"
RESULTS[REFERENCE] = {
    "person":     torch_person,
    "keypoints":  torch_person["keypoints_2d"],
    "pck":        torch_pck,
    "latency_ms": torch_model.last_inference_time_ms,
}

print(f"Latency  : {torch_model.last_inference_time_ms:8.1f} ms")
print(f"PCK@{PCK_THRESHOLD} : {torch_pck:8.2f} %   "
      f"({int(torch_correct.sum())}/{int(torch_valid.sum())} visible keypoints correct)")

In [ ]:
data.show_row(
    list(zip(["2D keypoints", "3D mesh — front", "3D mesh — side"],
             data.render_views(IMG_BGR, torch_person, MESH_FACES))),
    suptitle=f"{REFERENCE} (reference)  —  PCK@{PCK_THRESHOLD} = {torch_pck:.2f}%",
)

In [ ]:
# Free the multi-GB PyTorch model before loading the OpenVINO pipelines.
# Predictions are already copied into RESULTS as plain NumPy.
del torch_model, outputs, pred
free_memory()
print("PyTorch model released.")

## Convert the model to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

`sam3d_torch.py` traces each PyTorch sub-module with `torch.jit.trace` and
converts it with `ov.convert_model`. The pipeline uses several IRs rather
than one graph, because the decoder runs an *iterative MHR feedback loop* that
cannot be captured as a single static graph:

| Group | Contents |
|---|---|
| `backbone/` | DINOv3 ViT-H/16+ (ImageNet normalization baked in) |
| `iterative/` | 6 decoder layers, final norm, pose/camera heads, token MLPs |
| `iterative/` (aux) | `grid_sample`, `camera_projection`, `full_to_crop` — stateless ops |
| `mhr/` | dense mesh + skeleton head (`DenseMHR`, a sparse-op-free rewrite) |
| `mask_encoder/` | mask-conditioning CNN |
| `iterative/buffers/` | static `.npz` tensors (init pose, embeddings, PCA bases) |

Precision: `fp16` stores weights as FP16, `int8` applies NNCF weight-only
INT8 compression (`nncf.compress_weights`) — no calibration data is needed and
activations stay floating point.

In [ ]:
from sam3d_ov import Sam3DBodyOpenVINO


def export_ir(precision):
    """Export the full pipeline to OpenVINO IR at `precision` (skips if present)."""
    out_dir = OV_MODEL_ROOT / precision
    if Sam3DBodyOpenVINO.is_available(out_dir, precision) and not FORCE_EXPORT:
        print(f"[skip] {precision.upper():5s} IR already complete at {out_dir}")
        return

    print(f"[export] {precision.upper()} -> {out_dir}  (several minutes)")
    subprocess.run(
        [sys.executable, str(NOTEBOOK_DIR / "sam3d_torch.py"),
         "--precision", precision,
         "--output_dir", str(OV_MODEL_ROOT),
         "--checkpoint", str(CHECKPOINT),
         "--mhr_path", str(MHR_PATH)],
        check=True,
    )
    print(f"[export] {precision.upper()} done.")


for precision in PRECISIONS:
    export_ir(precision)

In [ ]:
# --- Report the on-disk footprint of each precision --------------------------
print(f"{'Precision':<10} {'Backbone':>12} {'MHR':>10} {'Total IR':>12}")
print("-" * 46)

IR_SIZES = {}
for precision in PRECISIONS:
    d = OV_MODEL_ROOT / precision
    total = sum(f.stat().st_size for f in d.rglob("*.bin")) / 1024**2
    total += sum(f.stat().st_size for f in d.rglob("*.xml")) / 1024**2
    backbone = (d / "backbone" / f"backbone_{precision}.bin").stat().st_size / 1024**2
    mhr = (d / "mhr" / f"mhr_{precision}.bin").stat().st_size / 1024**2
    IR_SIZES[precision] = total
    print(f"{precision.upper():<10} {backbone:>10.0f} MB {mhr:>8.0f} MB {total:>10.0f} MB")

if "fp16" in IR_SIZES and "int8" in IR_SIZES:
    print(f"\nINT8 is {IR_SIZES['fp16'] / IR_SIZES['int8']:.2f}x smaller than FP16 on disk.")

## Run OpenVINO inference on Intel GPU
[back to top ⬆️](#Table-of-contents:)

`Sam3DBodyOpenVINO` reproduces the PyTorch pipeline and every heavy op
runs as an OpenVINO model on the GPU.

In [ ]:
def run_openvino(precision, device=OV_DEVICE):
    """Compile the OV pipeline at `precision` and infer on the sample person."""
    print(f"\n=== OpenVINO {precision.upper()} on {device} ===")
    pipe = Sam3DBodyOpenVINO(OV_MODEL_ROOT / precision, device=device, precision=precision)
    pipe.warmup(n=1)  # JIT-compile kernels so the timing is steady-state

    t0 = time.perf_counter()
    out = pipe.infer_single(IMG_RGB, BBOX_XYXY, focal_length=FOCAL)
    latency_ms = (time.perf_counter() - t0) * 1000

    person = {
        "keypoints_2d": out["j2d"],      # [70, 2]
        "bbox":         BBOX_XYXY,
        "vertices":     out["verts"],    # [V, 3]
        "cam_t":        out["cam_t"],
        "focal_length": out["focal_length"],
    }
    pck, correct, valid = compute_person_pck(person["keypoints_2d"], GT_ANN, PCK_THRESHOLD)

    print(f"Latency  : {latency_ms:8.1f} ms")
    print(f"PCK@{PCK_THRESHOLD} : {pck:8.2f} %   "
          f"({int(correct.sum())}/{int(valid.sum())} visible keypoints correct)")

    del pipe
    free_memory()
    return {"person": person, "keypoints": person["keypoints_2d"],
            "pck": pck, "latency_ms": latency_ms}


for precision in PRECISIONS:
    RESULTS[f"OpenVINO {precision.upper()}"] = run_openvino(precision)

In [ ]:
for name in [f"OpenVINO {p.upper()}" for p in PRECISIONS]:
    r = RESULTS[name]
    data.show_row(
        list(zip(["2D keypoints", "3D mesh — front", "3D mesh — side"],
                 data.render_views(IMG_BGR, r["person"], MESH_FACES))),
        suptitle=f"{name} on {OV_DEVICE}  —  PCK@{PCK_THRESHOLD} = {r['pck']:.2f}%",
    )

## Compare PyTorch and OpenVINO
[back to top ⬆️](#Table-of-contents:)

1. **Visually** — skeletons and meshes side by side.
2. **Numerically** — PCK@0.05 metric for each backend.

In [ ]:
# --- Side-by-side skeletons ---------------------------------------------------
data.show_row(
    [(f"{name}\nPCK={r['pck']:.2f}%", data.render_views(IMG_BGR, r["person"], MESH_FACES)[0])
     for name, r in RESULTS.items()],
    suptitle="2D keypoints — PyTorch reference vs OpenVINO",
)

# --- Side-by-side mesh overlays ----------------------------------------------
data.show_row(
    [(name, data.render_views(IMG_BGR, r["person"], MESH_FACES)[1]) for name, r in RESULTS.items()],
    suptitle="3D mesh overlay — PyTorch reference vs OpenVINO",
)

In [ ]:
# --- Numerical agreement ------------------------------------------------------
ref_kpts = to_numpy(RESULTS[REFERENCE]["keypoints"])[COCO17_TO_MHR70]

# PCK tolerance in pixels: a deviation below this cannot flip a keypoint's verdict.
TOLERANCE_PX = data.pck_tolerance_px(GT_ANN["bbox"], PCK_THRESHOLD)


header = (f"{'Backend':<18} {'PCK@0.05':>9} {'Latency':>10}")

print(header)
print("-" * len(header))

for name, r in RESULTS.items():
    kpts = to_numpy(r["keypoints"])[COCO17_TO_MHR70]
    dev = np.linalg.norm(kpts - ref_kpts, axis=-1)  # per-keypoint pixel error
    tag = "  (reference)" if name == REFERENCE else ""
    print(f"{name:<18} {r['pck']:>8.2f}% {r['latency_ms']:>8.0f}ms{tag}")



## Summary
[back to top ⬆️](#Table-of-contents:)

**What this notebook demonstrated**

1. The reference PyTorch SAM 3D Body model runs on a sample image on CPU or Intel XPU (if chosen).
2. The pipeline converts to OpenVINO IR at FP16 and INT8.
3. Both IRs run on the **Intel GPU** and produce
   keypoints that agree with PyTorch measured via the PCK@0.05 metric and and the meshes produced are visually indistinguishable.


**Running on your own image.** `data.make_sample()` turns any image plus a COCO-style
annotation into the dict used above.

```python
import cv2, sam3d_data as data
from sam3d_ov import Sam3DBodyOpenVINO

pipe = Sam3DBodyOpenVINO(OV_MODEL_ROOT / "fp16", device="GPU", precision="fp16")
pipe.warmup()

sample = data.make_sample(cv2.imread("my_photo.jpg"),
                          {"bbox": [x, y, w, h], "keypoints": [0] * 51})
out = pipe.infer_single(sample["img_rgb"], sample["bbox_xyxy"],
                        focal_length=sample["focal_length"])
person = {"keypoints_2d": out["j2d"], "bbox": sample["bbox_xyxy"]}
data.show_row([("prediction", data.render_views(sample["img_bgr"], person, None)[0])])
```

**Useful links**

- [SAM 3D Body](https://github.com/facebookresearch/sam-3d-body)
- [OpenVINO documentation](https://docs.openvino.ai/)
- [OpenVINO notebooks](https://github.com/openvinotoolkit/openvino_notebooks)